## Save LLM summarization outputs to the master tables (for llm_summarization_gpt v10.x)

Reads the files the v10 notebook writes and loads two Delta tables. It no longer gates, scrubs or splits anything: the v10 notebook already applied the QA gate (client file versus quarantine file), scrubbed the text for Excel, and built the one-row-per-scheme table. This notebook validates those files, writes the rows for the scoring date, and prints what it wrote.

What changed from the v9 version of this notebook
- Source: the v10 client and schemes CSVs (`all_providers_llm_based_summary_{date}.csv`, `all_providers_llm_based_summary_schemes_{date}.csv`), not the parquet checkpoint folder. The v10 checkpoint is a JSONL used only for resume.
- No second QA gate. The v9 gate `QA_Flags == 'pass'` would drop every Python note (evidence, low-volume, patient-mix rows carry their type in QA_Flags, not 'pass'). v10's gate is `QA_Status`; the client file holds pass, evidence_only, patient_mix, low_volume and evidence_fallback rows, one per referred provider-category.
- No scheme splitting here. The v9 regex `^(.+?)\s*\(([^)]+)\)` would turn "Not confirmed (see Possible_Schemes)" into a scheme called "Not confirmed". v10 writes the schemes file from confirmed candidates only (`... (high|medium|low evidence)`).
- Duplicates are judged on (provider, category), not provider alone: a provider referred in two categories keeps both rows.
- The v9 columns Signal_Concentration, Perspective_Agreement, Dominant_Perspective do not exist in v10 and are not written; the new columns (Note_Type, QA_Status, Scheme_Status, Scheme_Status_Reason, Possible_Schemes, Scheme_Strength, Main_Pattern_Stability) are added with mergeSchema. No version columns reach the tables: Notebook_Version and Dictionary_Version stay in the CSVs, the audit file and the trust report, and this notebook only uses them to refuse a file from another architecture.
- Re-running for the same scoring date replaces that date's rows (REPLACE_SCORING_DATE), so the tables never carry two copies of one run. Set it to False for the v9 append behavior.

Two switches in the configuration cell decide what the summarization table holds: WRITE_ALL_CLIENT_ROWS (every referred provider-category with its note type) or narratives only, as v9 did.

In [ ]:
import re
import os
import json
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 250)

##### Configuration

In [ ]:
provider_id = 'Servicing_Provider_ID'
ROOT = 'sedo.senou@azahcccs.gov'
DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/all_projects_predictions/data'

with open(f'{DATA_ROOT}/scoring_date.txt', 'r') as f:
    scoring_date = f.read().strip()

master_table_llm_summarization = "main.prod_results.master_anomaly_detection_referrals_and_llm_summarization"
master_table_llm_schemes = "main.prod_results.master_anomaly_detection_llm_schemes"

# Files the v10 notebook wrote for this scoring date
CLIENT_FILE = f'{DATA_ROOT}/all_providers_llm_based_summary_{scoring_date}.csv'
SCHEMES_FILE = f'{DATA_ROOT}/all_providers_llm_based_summary_schemes_{scoring_date}.csv'
QUARANTINE_FILE = f'{DATA_ROOT}/llm_summary_quarantine_{scoring_date}.csv'
FULL_FILE = f'{DATA_ROOT}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony_{scoring_date}.csv'

EXPECTED_NOTEBOOK_PREFIX = 'v10'      # refuse files from another architecture
WRITE_ALL_CLIENT_ROWS = True          # True: every referred provider-category with its note type; False: narratives only (v9 behavior)
REPLACE_SCORING_DATE = True           # True: delete this scoring date's rows before writing (idempotent re-runs); False: append

# Columns written to the summarization table (all strings; mergeSchema adds the new ones to the existing table).
# No version columns: the notebook and dictionary versions stay in the CSVs, the audit file and the trust report,
# never in the end-user tables.
SUMMARY_COLS = [provider_id, 'Category', 'Scoring_Date', 'LLM_Summary', 'Note_Type', 'QA_Status', 'Detected_Schemes',
                'Scheme_Status', 'Scheme_Status_Reason', 'Possible_Schemes', 'Scheme_Strength', 'Scheme_Rationale',
                'Narrative_Features', 'Main_Pattern_Stability', 'Provider_Anomaly_Score', 'Category_Rank']
SCHEME_COLS = [provider_id, 'Category', 'Scoring_Date', 'Detected_Scheme', 'Evidence_Strength']
print(f'Scoring date {scoring_date} | write all client rows: {WRITE_ALL_CLIENT_ROWS} | replace date: {REPLACE_SCORING_DATE}')

##### Load and validate the v10 files

In [ ]:
# ── Load the v10 files and validate them before anything touches a table ──
for p in (CLIENT_FILE, SCHEMES_FILE):
    if not os.path.exists(p):
        raise FileNotFoundError(f'{p} not found: run llm_summarization_gpt v10 for {scoring_date} first')
client_df = pd.read_csv(CLIENT_FILE, dtype=str, encoding='utf-8-sig').fillna('')
schemes_df = pd.read_csv(SCHEMES_FILE, dtype=str, encoding='utf-8-sig').fillna('')
quarantine_df = pd.read_csv(QUARANTINE_FILE, dtype=str, encoding='utf-8-sig').fillna('') if os.path.exists(QUARANTINE_FILE) else pd.DataFrame()

missing = [c for c in SUMMARY_COLS + ['Notebook_Version', 'Dictionary_Version'] if c not in client_df.columns and c != 'Scoring_Date']
if missing:
    raise ValueError(f'Client file lacks columns {missing}: it was not written by the v10 notebook this loader expects')
versions = sorted(client_df['Notebook_Version'].unique())
if not all(v.startswith(EXPECTED_NOTEBOOK_PREFIX) for v in versions):
    raise ValueError(f'Notebook_Version {versions} is not {EXPECTED_NOTEBOOK_PREFIX}.x')
dates = sorted(client_df['Scoring_Date'].unique())
if dates != [scoring_date]:
    raise ValueError(f'Scoring_Date in the client file is {dates}, expected {scoring_date}')

# One row per referred provider-category (a provider can be referred in two categories)
dup = client_df.duplicated(subset=[provider_id, 'Category'], keep=False)
if dup.any():
    print(f'WARNING: {int(dup.sum())} duplicated provider-category rows; keeping the last of each')
    client_df = client_df.drop_duplicates(subset=[provider_id, 'Category'], keep='last')
bad_status = set(client_df['QA_Status']) - {'pass', 'evidence_only', 'patient_mix', 'low_volume', 'evidence_fallback'}
if bad_status:
    raise ValueError(f'Client file carries non-client QA_Status values {bad_status}: the v10 gate did not run')

print(f'Client rows: {len(client_df)} | versions {versions} | dictionary {sorted(client_df["Dictionary_Version"].unique())}')
print('Note types:', client_df['Note_Type'].value_counts().to_dict())
print('QA status:', client_df['QA_Status'].value_counts().to_dict())
print('Scheme status:', client_df['Scheme_Status'].value_counts().to_dict())
print(f'Schemes file: {len(schemes_df)} rows for {schemes_df[provider_id].nunique()} providers | strengths {schemes_df["Evidence_Strength"].value_counts().to_dict()}')
print(f'Quarantine file: {len(quarantine_df)} rows' + (f' | flags {quarantine_df["QA_Status"].value_counts().to_dict()}' if len(quarantine_df) else ''))

##### Review quarantined narratives

In [ ]:
# ── Review: quarantined narratives (their providers already have an evidence fallback in the client file) ──
if len(quarantine_df):
    print(f'{len(quarantine_df)} quarantined narratives (not written to the tables); first 5:')
    for _, row in quarantine_df.head(5).iterrows():
        print('\n' + '=' * 60)
        print(f"Provider: {row[provider_id]} | Category: {row['Category']} | QA: {row['QA_Flags']}")
        print(f"Summary preview: {row['LLM_Summary'][:300]}...")
    fb = client_df[client_df['Note_Type'] == 'evidence_fallback']
    print(f'\nEvidence fallbacks in the client file for them: {len(fb)}')
else:
    print('No quarantined narratives for this scoring date.')

##### Rows for the tables

In [ ]:
# ── Rows for the tables ──
MOJIBAKE = {'â€¯': ' ', 'â€"': '-', 'â€“': '-', 'â€œ': '"', 'â€\x9d': '"', 'â€™': "'", 'â€˜': "'", 'Â ': ' ', 'Â': ''}


def tidy(text):
    """Belt and braces: the v10 notebook already scrubbed; this catches a file that went through Excel on the way."""
    if not isinstance(text, str):
        return text
    for bad, good in MOJIBAKE.items():
        text = text.replace(bad, good)
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    return text.strip()


rows = client_df if WRITE_ALL_CLIENT_ROWS else client_df[client_df['Note_Type'] == 'narrative']
delta_summary = rows[SUMMARY_COLS].copy()
for c in ('LLM_Summary', 'Detected_Schemes', 'Scheme_Rationale', 'Possible_Schemes'):
    delta_summary[c] = delta_summary[c].map(tidy)
delta_summary['Scoring_Date'] = pd.to_datetime(delta_summary['Scoring_Date'], errors='coerce').dt.strftime('%Y-%m-%d')
delta_summary = delta_summary.astype(str)

delta_schemes = schemes_df.copy()
delta_schemes['Scoring_Date'] = pd.to_datetime(delta_schemes['Scoring_Date'], errors='coerce').dt.strftime('%Y-%m-%d')
delta_schemes = delta_schemes[SCHEME_COLS].astype(str)
# every scheme row must point at a client row with a confirmed status
_keys = set(zip(client_df[provider_id], client_df['Category']))
_orphans = [k for k in zip(delta_schemes[provider_id], delta_schemes['Category']) if k not in _keys]
if _orphans:
    raise ValueError(f'{len(_orphans)} scheme rows have no client row: {_orphans[:5]}')
print(f'Summarization rows to write: {len(delta_summary)} | scheme rows to write: {len(delta_schemes)}')

##### Save to Delta tables

In [ ]:
# ── Write to Delta: create, or replace this scoring date's rows, then append ──
spark = SparkSession.builder.appName("Referral_Processing").getOrCreate()


def write_table(df, cols, table):
    schema = StructType([StructField(c, StringType(), True) for c in cols])
    df_spark = spark.createDataFrame(df[cols], schema=schema)
    if not spark.catalog.tableExists(table):
        df_spark.write.mode('overwrite').saveAsTable(table)
        print(f'{table}: created with {len(df)} rows')
        return
    if REPLACE_SCORING_DATE:
        before = spark.sql(f"SELECT COUNT(*) AS n FROM {table} WHERE Scoring_Date = '{scoring_date}'").collect()[0]['n']
        if before:
            spark.sql(f"DELETE FROM {table} WHERE Scoring_Date = '{scoring_date}'")
            print(f'{table}: removed {before} rows already there for {scoring_date}')
    df_spark.write.mode('append').option('mergeSchema', 'true').saveAsTable(table)
    print(f'{table}: appended {len(df)} rows')


if delta_summary.empty:
    print('No summaries to write')
else:
    write_table(delta_summary, SUMMARY_COLS, master_table_llm_summarization)
if delta_schemes.empty:
    print('No schemes to write (no confirmed scheme for this scoring date)')
else:
    write_table(delta_schemes, SCHEME_COLS, master_table_llm_schemes)

##### Verify

In [ ]:
# ── Verify what the tables hold for this scoring date ──
for table, expected in ((master_table_llm_summarization, len(delta_summary)), (master_table_llm_schemes, len(delta_schemes))):
    if spark.catalog.tableExists(table):
        n = spark.sql(f"SELECT COUNT(*) AS n FROM {table} WHERE Scoring_Date = '{scoring_date}'").collect()[0]['n']
        print(f'{table}: {n} rows for {scoring_date} (expected {expected})' + ('' if n == expected else '  <-- CHECK'))
if spark.catalog.tableExists(master_table_llm_summarization):
    print(spark.sql(f"SELECT Note_Type, Scheme_Status, COUNT(*) AS n FROM {master_table_llm_summarization} "
                    f"WHERE Scoring_Date = '{scoring_date}' GROUP BY Note_Type, Scheme_Status ORDER BY n DESC").toPandas().to_string(index=False))

##### Sample output

In [ ]:
# ── Sample output ──
for _, row in delta_summary[delta_summary['Note_Type'] == 'narrative'].head(3).iterrows():
    print('\n' + '=' * 80)
    print(f"Provider: {row[provider_id]} | Category: {row['Category']} | Note: {row['Note_Type']} | QA: {row['QA_Status']}")
    print(f"Schemes: {row['Detected_Schemes']} | Status: {row['Scheme_Status']} ({row['Scheme_Status_Reason']})")
    print(f"\n{row['LLM_Summary']}")